In [ ]:
import os
import torch
from torch.utils.data import DataLoader
from torch.optim import Adam
from torch.cuda.amp import GradScaler

# Import utils
from utils.Logger import Logger
from utils.Seed import set_seed
from utils.Splitter import stratified_split
from classes.FeatureDataset.WaveformFeatureDataset import WaveformFeatureDataset
from classes.FeatureDataset.ListDataset import ListDataset

# Import XLSR-SLS components
from classes.models.XLSR_SLS.model_XLSR_SLS import XLSRSLS
from classes.models.XLSR_SLS.trainer_XLSR_SLS import (
    test_xlsr_sls,
    load_model_xlsr_sls
)

seed = 42
set_seed(42)

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = ""

device = "cuda" if torch.cuda.is_available() else "cpu"

batch_size = 32
learning_rate = 0.000001
epochs = 30

main_keys = [
    "unseen all samples (UnID)"
]

# XLSR-SLS

## Load Test Data

In [ ]:
# samples will be stored here:
xlsr_sls_test_samples = {
    "unseen all samples (UnID)": [],
}

# unseen spoof datasets
xlsr_sls_spoof_dupdub_notindataset_tts_dir = "test_preprocessed_data/waveform/Spoof/TTS/DupDub-NotInDataset"

# Bonafide datasets
xlsr_sls_bonafide_commonvoice_dir = "preprocessed_data/waveform/Bonafide/CommonVoice"
xlsr_sls_bonafide_prosa_dir = "preprocessed_data/waveform/Bonafide/Prosa"

## ------------------------------------
## UNSEEN SPOOF DATASETS
## ------------------------------------
# DupDub NotInDataset TTS
if os.path.exists(xlsr_sls_spoof_dupdub_notindataset_tts_dir):
    xlsr_sls_spoof_dupdub_notindataset_tts_dataset = WaveformFeatureDataset(xlsr_sls_spoof_dupdub_notindataset_tts_dir, force_label=0)
    xlsr_sls_spoof_dupdub_notindataset_tts_list = ListDataset([(features, 0) for features, _ in xlsr_sls_spoof_dupdub_notindataset_tts_dataset.samples])
    xlsr_sls_test_samples["unseen all samples (UnID)"].extend(xlsr_sls_spoof_dupdub_notindataset_tts_list)

    print(f"Loaded {len(xlsr_sls_spoof_dupdub_notindataset_tts_list)} samples from {xlsr_sls_spoof_dupdub_notindataset_tts_dir}")
else:
    print(f"Warning: Directory not found: {xlsr_sls_spoof_dupdub_notindataset_tts_dir}")

### Bonafides
# Bonafide CommonVoice
if os.path.exists(xlsr_sls_bonafide_commonvoice_dir):
    dataset_commonvoice = WaveformFeatureDataset(xlsr_sls_bonafide_commonvoice_dir, force_label=1)
    xlsr_sls_bonafide_commonvoice = ListDataset([(features, 1) for features, _ in dataset_commonvoice.samples])
    t_c, v_c, te_c = stratified_split(xlsr_sls_bonafide_commonvoice, splits=(0.7, 0.15, 0.15), seed=seed)

    xlsr_sls_bonafide_commonvoice_list = ListDataset([xlsr_sls_bonafide_commonvoice[i] for i in range(len(te_c))])
    xlsr_sls_test_samples["unseen all samples (UnID)"].extend(xlsr_sls_bonafide_commonvoice_list)

    print(f"Loaded {len(xlsr_sls_bonafide_commonvoice_list)} samples from {xlsr_sls_bonafide_commonvoice_dir}")
else:
    print(f"Warning: Directory not found: {xlsr_sls_bonafide_commonvoice_dir}")

# Bonafide Prosa
if os.path.exists(xlsr_sls_bonafide_prosa_dir):
    dataset_prosa = WaveformFeatureDataset(xlsr_sls_bonafide_prosa_dir, force_label=1)
    xlsr_sls_bonafide_prosa = ListDataset([(features, 1) for features, _ in dataset_prosa.samples])
    t_p, v_p, te_p = stratified_split(xlsr_sls_bonafide_prosa, splits=(0.7, 0.15, 0.15), seed=seed)

    xlsr_sls_bonafide_prosa_list = ListDataset([xlsr_sls_bonafide_prosa[i] for i in range(len(te_p))])
    xlsr_sls_test_samples["unseen all samples (UnID)"].extend(xlsr_sls_bonafide_prosa_list)

    print(f"Loaded {len(xlsr_sls_bonafide_prosa_list)} samples from {xlsr_sls_bonafide_prosa_dir}")
else:
    print(f"Warning: Directory not found: {xlsr_sls_bonafide_prosa_dir}")

print("-----------------------------------------------")
print(f"Total unseen UnID samples: {len(xlsr_sls_test_samples['unseen all samples (UnID)'])}")
print("-----------------------------------------------")

xlsr_sls_test_samples_dataloaders = {
    key: DataLoader(value, batch_size=batch_size, shuffle=False, num_workers=4)
    for key, value in xlsr_sls_test_samples.items()
}

## Test XLSR-SLS Model

In [ ]:
# Model configuration (same as training)
xlsr_sls_model_config = {
    'cp_path': 'xlsr2_300m.pt',  # Path to pretrained XLSR model
    'fine_tune_ssl': True        # Whether to fine-tune SSL model
}

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# --- Set up model, optimizer, and scaler ---
model = XLSRSLS(xlsr_sls_model_config, device).to(device)
optimizer = Adam(model.parameters(), lr=learning_rate)
scaler = GradScaler()

# --- Load the trained model ---
checkpoint_path = r"pretrained_weights/waveform/XLSR_SLS/xlsr_sls_waveform-ep_30-bs_32-lr_1e-06.pth"

print(f"\nLoading model from: {checkpoint_path}")
start_epoch = load_model_xlsr_sls(
    model, optimizer, scaler,
    path=checkpoint_path,
    device=device
)

print(f"Loaded checkpoint from epoch {start_epoch}")

# Count parameters
nb_params = sum([param.view(-1).size()[0] for param in model.parameters() if param.requires_grad])
print(f'Number of trainable parameters: {nb_params:,}')

In [ ]:
# Test the model
for key, test_sample in xlsr_sls_test_samples_dataloaders.items():
    print(f"\n{'='*60}")
    print(f"Testing {key}")
    print(f"{'='*60}")
    
    if key in main_keys:
        predictions, targets, metrics = test_xlsr_sls(model, test_sample, device=device)
        
        # Print summary
        print(f"\n--- Results Summary ---")
        print(f"Accuracy: {metrics['accuracy']:.2f}%")
        print(f"Balanced Accuracy: {metrics['balanced_accuracy']:.4f}")
        print(f"Precision: {metrics['precision']:.4f}")
        print(f"Recall: {metrics['recall']:.4f}")
        print(f"F1 Score: {metrics['f1']:.4f}")
        print(f"F2 Score: {metrics['f2']:.4f}")
        print(f"EER: {metrics['eer']:.4f}")
        print(f"actDCF: {metrics['actDCF']:.4f}")
        print(f"minDCF: {metrics['minDCF']:.4f}")
        print(f"CLLR: {metrics['cllr']:.4f}")
    
    torch.cuda.empty_cache()

print(f"\n{'='*60}")
print("Testing completed!")
print(f"{'='*60}")